In [1]:
# importing necessary libraries
import pandas as pd
import numpy as np
import ast


In [2]:
# retrieve the data
df = pd.read_csv('../../data/raw/tmdb/tmdb_raw.csv')
df.head(5)

,tmdb_id,title,original_title,popularity,vote_average,vote_count,release_date,original_language,genre_ids,adult
0,969681,Spider-Man: Brand New Day,Spider-Man: Brand New Day,653.7739,7.900,2849,2026-07-29,en,"[878, 28, 12]",False
1,1423191,Resident Evil,Resident Evil,584.5500,7.307,374,2026-09-16,en,"[27, 878, 12]",False
2,1101383,The End of Oak Street,The End of Oak Street,485.8521,7.006,1455,2026-08-12,en,"[878, 9648, 53]",False
3,1204680,Coyote vs. Acme,Coyote vs. Acme,409.4076,7.556,502,2026-08-20,en,"[35, 12, 10751]",False
4,1368337,The Odyssey,The Odyssey,346.9968,8.019,3878,2026-07-15,en,"[12, 28, 14]",False


In [3]:
df.columns

Index(['tmdb_id', 'title', 'original_title', 'popularity', 'vote_average',
       'vote_count', 'release_date', 'original_language', 'genre_ids',
       'adult'],
      dtype='str')

In [4]:
df.dtypes

tmdb_id                int64
title                    str
original_title           str
popularity           float64
vote_average         float64
vote_count             int64
release_date             str
original_language        str
genre_ids                str
adult                   bool
dtype: object

## let's inspect each column and treat problems one by one

In [5]:
df.isnull().sum()

tmdb_id              0
title                0
original_title       0
popularity           0
vote_average         0
vote_count           0
release_date         0
original_language    0
genre_ids            0
adult                0
dtype: int64

### checking for duplicate movies (same tmdb_id appearing more than once)

In [6]:
df.duplicated(subset="tmdb_id").sum()

np.int64(7)

In [7]:
nb_avant = len(df)
df = df.drop_duplicates(subset="tmdb_id", keep="first")
print(f"Doublons supprimes : {nb_avant - len(df)}")

Doublons supprimes : 7


## release_date: converting from text to a real date, and extracting the year
for consistency with the other datasets (imdb, rotten tomatoes)

In [8]:
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year
df[["release_date", "release_year"]].head(10)

,release_date,release_year
0,2026-07-29,2026
1,2026-09-16,2026
2,2026-08-12,2026
3,2026-08-20,2026
4,2026-07-15,2026
5,2026-05-21,2026
6,2026-08-19,2026
7,2026-07-08,2026
8,2026-09-09,2026
9,2026-06-17,2026


In [9]:
# checking if any release_date failed to convert (would show up as NaT)
df["release_date"].isnull().sum()

np.int64(0)

genre_ids is stored as text after reading the csv (e.g. "[28, 12]") instead of a real list
we convert it back into a python list using ast.literal_eval

In [10]:
df["genre_ids"] = df["genre_ids"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else [])
df["genre_ids"].head(5)

0      [878, 28, 12]
1      [27, 878, 12]
2    [878, 9648, 53]
3    [35, 12, 10751]
4       [12, 28, 14]
Name: genre_ids, dtype: object

additional check: vote_average should always be between 0 and 10 on TMDB
if not, that would indicate corrupted data from the API

In [11]:
print("Min vote_average:", df["vote_average"].min())
print("Max vote_average:", df["vote_average"].max())
print("Any negative popularity?", (df["popularity"] < 0).any())
print("Any negative vote_count?", (df["vote_count"] < 0).any())

Min vote_average: 3.5
Max vote_average: 9.16
Any negative popularity? False
Any negative vote_count? False


checking which languages are present, to spot any anomaly (unexpected codes)

In [12]:
df["original_language"].value_counts()

original_language
en    312
fr     59
es     44
it     22
hi     18
zh     13
ja     13
ko     12
de     11
ta      9
pt      8
te      4
id      3
th      3
ru      2
pl      2
tl      2
sv      2
cn      2
ar      2
ml      2
no      1
kn      1
fi      1
ca      1
tr      1
el      1
ro      1
af      1
hu      1
nl      1
Name: count, dtype: int64

final verification of data types before saving

In [13]:
df["tmdb_id"] = df["tmdb_id"].astype(int)
df["adult"] = df["adult"].astype(bool)
df.dtypes

tmdb_id                       int64
title                           str
original_title                  str
popularity                  float64
vote_average                float64
vote_count                    int64
release_date         datetime64[us]
original_language               str
genre_ids                    object
adult                          bool
release_year                  int32
dtype: object

In [14]:
# reordering columns for better readability
df = df[[
    "tmdb_id", "title", "original_title", "release_date", "release_year",
    "genre_ids", "popularity", "vote_average", "vote_count",
    "original_language", "adult"
]]
df.head(5)

,tmdb_id,title,original_title,release_date,release_year,genre_ids,popularity,vote_average,vote_count,original_language,adult
0,969681,Spider-Man: Brand New Day,Spider-Man: Brand New Day,2026-07-29,2026,"[878, 28, 12]",653.7739,7.900,2849,en,False
1,1423191,Resident Evil,Resident Evil,2026-09-16,2026,"[27, 878, 12]",584.5500,7.307,374,en,False
2,1101383,The End of Oak Street,The End of Oak Street,2026-08-12,2026,"[878, 9648, 53]",485.8521,7.006,1455,en,False
3,1204680,Coyote vs. Acme,Coyote vs. Acme,2026-08-20,2026,"[35, 12, 10751]",409.4076,7.556,502,en,False
4,1368337,The Odyssey,The Odyssey,2026-07-15,2026,"[12, 28, 14]",346.9968,8.019,3878,en,False


In [16]:
import os
os.makedirs("../../data/processed", exist_ok=True)

print(df.shape)
df.to_csv("../../data/processed/tmdb_clean.csv", index=False)
print("Saved to data/processed/tmdb_clean.csv")

(555, 11)
Saved to data/processed/tmdb_clean.csv
